# LeetCode 1759 — Count Number of Homogenous Substrings bằng Python

Link đề: https://leetcode.com/problems/count-number-of-homogenous-substrings/ · Độ khó: **Medium** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def countHomogenous(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Phân biệt **chuỗi con** (substring — liên tiếp) với dãy con; đếm số chuỗi con của một đoạn dài `L`.
2. Nhận ra cấu trúc **"khối ký tự giống nhau liên tiếp"** (run-length) và công thức `L·(L+1)/2`.
3. Viết lời giải một lượt `O(n)` bằng **biến đếm chuỗi đang chạy** (`cur`), và bằng `itertools.groupby`.
4. Xử lý **modulo** `10⁹ + 7` đúng chỗ.
5. Đối chiếu nhiều cách với vét cạn trên chuỗi ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Trả về **số chuỗi con đồng nhất** (homogenous) của `s` — chuỗi con mà **mọi ký tự đều giống nhau**. Vì đáp án có thể lớn, trả về theo modulo `10⁹ + 7`. Chuỗi con là dãy ký tự **liên tiếp** trong `s`; các chuỗi con ở **vị trí khác nhau** được đếm riêng dù nội dung giống nhau.

**Ví dụ 1:** `s = "abbcccaa"` → `13`: `"a"` ×3, `"aa"` ×1, `"b"` ×2, `"bb"` ×1, `"c"` ×3, `"cc"` ×2, `"ccc"` ×1 ⇒ `3+1+2+1+3+2+1 = 13`.

**Ví dụ 2:** `s = "xy"` → `2` (`"x"` và `"y"`).

**Ví dụ 3:** `s = "zzzzz"` → `15`.

**Ràng buộc:** `1 ≤ s.length ≤ 10⁵`, `s` chỉ gồm chữ thường.

In [1]:
import random
import time
from itertools import groupby

MOD = 10 ** 9 + 7
examples = [("abbcccaa", 13), ("xy", 2), ("zzzzz", 15)]
for s, exp in examples:
    print(f"s={s!r} -> kỳ vọng {exp}")

s='abbcccaa' -> kỳ vọng 13
s='xy' -> kỳ vọng 2
s='zzzzz' -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1 Chuỗi con và cách liệt kê

Chuỗi độ dài `n` có `n·(n+1)/2` chuỗi con (chọn đầu `i ≤ j` cuối): `s[i:j+1]`.

In [2]:
s = "abc"
subs = [s[i:j + 1] for i in range(len(s)) for j in range(i, len(s))]
print(subs, len(subs), "= 3*4/2")

['a', 'ab', 'abc', 'b', 'bc', 'c'] 6 = 3*4/2


### 3.2 Kiểm tra chuỗi đồng nhất

`len(set(t)) == 1` hoặc `t == t[0] * len(t)` hoặc `t.count(t[0]) == len(t)`.

In [3]:
for t in ["aaa", "aab", "z"]:
    print(t, len(set(t)) == 1, t == t[0] * len(t))

aaa True True
aab False False
z True True


### 3.3 Tổng `1 + 2 + … + L = L·(L+1)/2`

Một khối gồm `L` ký tự giống nhau có `L` chuỗi con độ dài 1, `L−1` chuỗi con độ dài 2, …, 1 chuỗi con độ dài `L` ⇒ tổng `L(L+1)/2`. Dùng `//` để chia nguyên (tích `L(L+1)` luôn chẵn).

In [4]:
for L in [1, 2, 3, 5, 10 ** 5]:
    print(L, L * (L + 1) // 2, sum(range(1, L + 1)) if L < 100 else "...")

1 1 1
2 3 3
3 6 6
5 15 15
100000 5000050000 ...


### 3.4 `itertools.groupby` — gom các phần tử liên tiếp bằng nhau

`groupby("abbccc")` sinh các cặp `(ký tự, iterator)` cho từng **khối liên tiếp**; độ dài khối là `sum(1 for _ in g)`.

In [5]:
print([(ch, sum(1 for _ in g)) for ch, g in groupby("abbcccaa")])

[('a', 1), ('b', 2), ('c', 3), ('a', 2)]


### 3.5 Modulo `10⁹ + 7`

Python có số nguyên lớn tùy ý nên không tràn số, nhưng đề yêu cầu trả kết quả **mod** `10⁹+7`. Có thể lấy mod một lần ở cuối, hoặc sau mỗi phép cộng (giữ số nhỏ — bắt buộc trong C++/Java).

In [6]:
x = 10 ** 5 * (10 ** 5 + 1) // 2
print(x, x % MOD)

5000050000 49965


## 4. Ý tưởng nền tảng / trực giác

Tách **Ví dụ 1** `"abbcccaa"` thành các khối liên tiếp: `a` (1), `bb` (2), `ccc` (3), `aa` (2).

Một chuỗi con đồng nhất **không thể** vắt qua hai khối (vì ký tự ở ranh giới khác nhau), nên nó nằm trọn trong một khối. Đếm riêng mỗi khối:

| khối | L | L(L+1)/2 |
|---|---|---|
| a | 1 | 1 |
| bb | 2 | 3 |
| ccc | 3 | 6 |
| aa | 2 | 3 |
| **tổng** | | **13** |

Lưu ý hai khối `a` (đầu và cuối) được tính **riêng** — đúng như đề đếm theo vị trí.

In [7]:
blocks = [(ch, len(list(g))) for ch, g in groupby("abbcccaa")]
print(blocks, "->", [L * (L + 1) // 2 for _, L in blocks], "tổng", sum(L * (L + 1) // 2 for _, L in blocks))

[('a', 1), ('b', 2), ('c', 3), ('a', 2)] -> [1, 3, 6, 3] tổng 13


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi con

**Ý tưởng:** với mỗi cặp `(i, j)`, cắt `s[i:j+1]` và kiểm tra đồng nhất.

In [8]:
def count_bruteforce(s):
    n = len(s); cnt = 0
    for i in range(n):
        for j in range(i, n):
            t = s[i:j + 1]
            if t == t[0] * len(t):
                cnt += 1
    return cnt % MOD

for s, exp in examples:
    got = count_bruteforce(s); print(got, "kỳ vọng", exp); assert got == exp

13 kỳ vọng 13
2 kỳ vọng 2
15 kỳ vọng 15


**Độ phức tạp:** `O(n²)` cặp × `O(n)` mỗi lần cắt/so ⇒ `O(n³)`. Với `n = 10⁵` là `10¹⁵` — **không thể**.

Cải tiến nhỏ (Cách 1'): cố định `i`, mở rộng `j` **chừng nào** `s[j] == s[i]` rồi dừng — mỗi chuỗi con đồng nhất được đếm `O(1)`, nhưng tổng vẫn `O(n²)` khi cả chuỗi là một khối (`"aaaa…"`).

In [9]:
def count_expand(s):
    n = len(s); cnt = 0
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            cnt += 1; j += 1
    return cnt % MOD

print([count_expand(s) for s, _ in examples])
t = time.perf_counter(); count_expand("a" * 3000); print(f"'a'*3000: {(time.perf_counter()-t)*1000:.0f} ms (tăng bậc hai)")

[13, 2, 15]


'a'*3000: 181 ms (tăng bậc hai)


**Nút thắt:** trong một khối dài `L`, ta đếm lại từng chuỗi con một — trong khi đáp án cho khối có **công thức** `L(L+1)/2`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (khối):** chuỗi con đồng nhất nằm trọn trong một khối ⇒ đáp án `= Σ L_k(L_k+1)/2` trên các khối.

**Insight 2 (đếm theo điểm kết thúc):** với mỗi vị trí `j`, số chuỗi con đồng nhất **kết thúc tại `j`** bằng độ dài `cur` của chuỗi ký tự giống `s[j]` kéo dài về bên trái tới `j`. Quy tắc:
- **Nếu** `s[j] == s[j−1]` **thì** `cur += 1`;
- **ngược lại** `cur = 1`;
- cộng `cur` vào đáp án.

Hai insight cho cùng kết quả vì `1 + 2 + … + L = L(L+1)/2`.

**Invariant:** sau khi xử lý `s[0..j]`, `ans` = số chuỗi con đồng nhất nằm trong `s[0..j]`, `cur` = độ dài khối chứa `j` tính tới `j`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tách khối bằng hai con trỏ + công thức

In [10]:
def count_blocks(s):
    n = len(s); i = 0; ans = 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        L = j - i
        ans += L * (L + 1) // 2
        i = j
    return ans % MOD

print([count_blocks(s) for s, _ in examples])

[13, 2, 15]


**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionBlocks:
    def countHomogenous(self, s: str) -> int:
        MOD = 10 ** 9 + 7
        n, i, ans = len(s), 0, 0
        while i < n:
            j = i
            while j < n and s[j] == s[i]:
                j += 1
            L = j - i
            ans = (ans + L * (L + 1) // 2) % MOD
            i = j
        return ans

print([SolutionBlocks().countHomogenous(s) for s, _ in examples])

[13, 2, 15]


**Độ phức tạp Cách 2:** mỗi vị trí được `j` đi qua đúng một lần ⇒ Time `O(n)`, Space `O(1)`.

### 7.2 Cách 3 — Một lượt với biến `cur` (đếm theo điểm kết thúc)

In [12]:
def count_streak(s):
    ans = cur = 0
    for j, ch in enumerate(s):
        cur = cur + 1 if j > 0 and ch == s[j - 1] else 1
        ans += cur
    return ans % MOD

print([count_streak(s) for s, _ in examples])

[13, 2, 15]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — `itertools.groupby` (Pythonic)

In [13]:
class SolutionGroupby:
    def countHomogenous(self, s: str) -> int:
        total = 0
        for _, g in groupby(s):
            L = sum(1 for _ in g)
            total += L * (L + 1) // 2
        return total % (10 ** 9 + 7)

print([SolutionGroupby().countHomogenous(s) for s, _ in examples])

[13, 2, 15]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)` (groupby duyệt lười). Code ngắn nhất, nhưng che mất cơ chế bên trong — nên hiểu Cách 2/3 trước.

### 7.4 Cách 5 — Quy hoạch động tường minh `dp[j]` (cầu nối tới các bài đếm khác)

Viết lại Cách 3 dưới dạng quy hoạch động để thấy rõ "trạng thái":
- `dp[j]` = số chuỗi con đồng nhất **kết thúc đúng tại** `j`.
- Cơ sở: `dp[0] = 1`.
- Chuyển: `dp[j] = dp[j−1] + 1` nếu `s[j] == s[j−1]`, ngược lại `dp[j] = 1`.
- Đáp án: `Σ dp[j]`.

Cách 3 chỉ là bản **tiết kiệm bộ nhớ** của DP này (chỉ cần `dp[j−1]` để tính `dp[j]`). Mẫu "dp theo điểm kết thúc" này dùng lại nguyên xi cho 413 Arithmetic Slices, 2348 Zero-Filled Subarrays, 1513 Substrings With Only 1s.

In [14]:
def count_dp(s):
    n = len(s)
    dp = [0] * n
    for j in range(n):
        dp[j] = dp[j - 1] + 1 if j > 0 and s[j] == s[j - 1] else 1
    return dp, sum(dp) % MOD

for s, exp in examples:
    dp, ans = count_dp(s)
    print(f"{s!r}: dp={dp} -> {ans} (kỳ vọng {exp})")
    assert ans == exp

'abbcccaa': dp=[1, 1, 2, 1, 2, 3, 1, 2] -> 13 (kỳ vọng 13)
'xy': dp=[1, 1] -> 2 (kỳ vọng 2)
'zzzzz': dp=[1, 2, 3, 4, 5] -> 15 (kỳ vọng 15)


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(n)` cho mảng `dp` (Cách 3 giảm còn `O(1)`).

> **Ghi chú về modulo ở ngôn ngữ khác:** trong C++/Java, `ans` kiểu `int` sẽ tràn khi vượt `2³¹ − 1 ≈ 2,1·10⁹` — đáp án thô ở đây lên tới `5·10⁹`. Vì vậy phải lấy mod **sau mỗi phép cộng** (hoặc dùng `long`). Trong Python không tràn số, nhưng lấy mod dần vẫn giữ số nhỏ và là thói quen tốt khi chuyển ngôn ngữ.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** `"abbcccaa"`:

| j | s[j] | giống s[j−1]? | cur | ans |
|---|---|---|---|---|
| 0 | a | — | 1 | 1 |
| 1 | b | không | 1 | 2 |
| 2 | b | có | 2 | 4 |
| 3 | c | không | 1 | 5 |
| 4 | c | có | 2 | 7 |
| 5 | c | có | 3 | 10 |
| 6 | a | không | 1 | 11 |
| 7 | a | có | 2 | **13** |

In [15]:
def trace_streak(s):
    ans = cur = 0
    for j, ch in enumerate(s):
        same = j > 0 and ch == s[j - 1]
        cur = cur + 1 if same else 1
        ans += cur
        ends = [s[j - t + 1:j + 1] for t in range(1, cur + 1)]
        print(f"j={j} s[j]={ch!r} giống trước={same!s:<5} cur={cur} ans={ans:>3} | chuỗi con kết thúc tại j: {ends}")
    return ans % MOD

assert trace_streak("abbcccaa") == 13

j=0 s[j]='a' giống trước=False cur=1 ans=  1 | chuỗi con kết thúc tại j: ['a']
j=1 s[j]='b' giống trước=False cur=1 ans=  2 | chuỗi con kết thúc tại j: ['b']
j=2 s[j]='b' giống trước=True  cur=2 ans=  4 | chuỗi con kết thúc tại j: ['b', 'bb']
j=3 s[j]='c' giống trước=False cur=1 ans=  5 | chuỗi con kết thúc tại j: ['c']
j=4 s[j]='c' giống trước=True  cur=2 ans=  7 | chuỗi con kết thúc tại j: ['c', 'cc']
j=5 s[j]='c' giống trước=True  cur=3 ans= 10 | chuỗi con kết thúc tại j: ['c', 'cc', 'ccc']
j=6 s[j]='a' giống trước=False cur=1 ans= 11 | chuỗi con kết thúc tại j: ['a']
j=7 s[j]='a' giống trước=True  cur=2 ans= 13 | chuỗi con kết thúc tại j: ['a', 'aa']


Trace Cách 2 (khối) trên **Ví dụ 3**:

In [16]:
def trace_blocks(s):
    i, ans, n = 0, 0, len(s)
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        L = j - i; ans += L * (L + 1) // 2
        print(f"khối s[{i}:{j}]={s[i:j]!r}, L={L}, đóng góp {L*(L+1)//2}, ans={ans}")
        i = j
    return ans % MOD

assert trace_blocks("zzzzz") == 15

khối s[0:5]='zzzzz', L=5, đóng góp 15, ans=15


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — một lượt, `O(n)` thời gian, `O(1)` bộ nhớ, lấy mod sau mỗi lần cộng.

In [17]:
class Solution:
    def countHomogenous(self, s: str) -> int:
        MOD = 10 ** 9 + 7
        ans = cur = 0
        prev = ""
        for ch in s:
            cur = cur + 1 if ch == prev else 1
            prev = ch
            ans = (ans + cur) % MOD
        return ans

for s, exp in examples:
    got = Solution().countHomogenous(s); print(got, exp); assert got == exp

13 13
2 2
15 15


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Mọi chuỗi con đồng nhất `s[i..j]` nằm trọn trong một khối tối đại. *Chứng minh:* nếu nó chứa hai vị trí kề nhau thuộc hai khối khác nhau thì hai ký tự đó khác nhau (định nghĩa ranh giới khối) — mâu thuẫn với đồng nhất.

**Bổ đề 2.** Khối độ dài `L` có đúng `L(L+1)/2` chuỗi con, và chúng đều đồng nhất.

⇒ Đáp án `= Σ_k L_k(L_k+1)/2` (Cách 2, 4).

**Cách 3:** phân hoạch các chuỗi con đồng nhất theo **điểm kết thúc** `j`. Những chuỗi kết thúc tại `j` là `s[j−t+1..j]` với `1 ≤ t ≤ cur_j`, trong đó `cur_j` là số ký tự liên tiếp bằng `s[j]` kết thúc tại `j` — đúng `cur_j` chuỗi. Quy tắc cập nhật `cur` tính đúng `cur_j` bằng quy nạp. Tổng `Σ_j cur_j` là đáp án. Lấy mod sau mỗi phép cộng không đổi kết quả cuối theo tính chất `(a + b) mod M = ((a mod M) + b) mod M`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê + kiểm tra | `O(n³)` | `O(n)` |
| 1'. Mở rộng từ mỗi `i` | `O(n²)` tệ nhất (`"aaa…"`) | `O(1)` |
| 2. Khối + công thức | `O(n)` | `O(1)` |
| 3. Biến `cur` | `O(n)` | `O(1)` |
| 4. `groupby` | `O(n)` | `O(1)` |

In [18]:
rng = random.Random(1759)
cases = {"toàn 'a'": "a" * 10 ** 5, "ngẫu nhiên ab": "".join(rng.choice("ab") for _ in range(10 ** 5))}
for name, s in cases.items():
    row = []
    for fname, f in [("blocks", count_blocks), ("streak", count_streak), ("submit", Solution().countHomogenous),
                     ("groupby", SolutionGroupby().countHomogenous)]:
        t = time.perf_counter(); r = f(s); row.append(f"{fname}={r} {(time.perf_counter()-t)*1000:.0f}ms")
    print(f"{name:<14}", " | ".join(row))

toàn 'a'       blocks=49965 3ms | streak=49965 6ms | submit=49965 5ms | groupby=49965 2ms
ngẫu nhiên ab  blocks=199393 8ms | streak=199393 6ms | submit=199393 5ms | groupby=199393 11ms


## 12. Bộ test

In [19]:
ALL = {"expand": count_expand, "dp": lambda s: count_dp(s)[1], "blocks": count_blocks, "streak": count_streak,
       "SolBlocks": SolutionBlocks().countHomogenous, "SolGroupby": SolutionGroupby().countHomogenous,
       "submit": Solution().countHomogenous}
def run_all(s):
    res = {k: f(s) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (s, res)
    return next(iter(res.values()))

for s, exp in examples:
    assert run_all(s) == exp
edge = [("a", 1), ("ab", 2), ("aa", 3), ("aba", 3), ("abab", 4), ("aabb", 6), ("zzzzzzzzzz", 55)]
for s, exp in edge:
    assert count_bruteforce(s) == exp and run_all(s) == exp
rng = random.Random(17590)
for _ in range(1500):
    s = "".join(rng.choice(rng.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])) for _ in range(rng.randint(1, 25)))
    assert run_all(s) == count_bruteforce(s), s
big = "a" * 10 ** 5
assert Solution().countHomogenous(big) == (10 ** 5 * (10 ** 5 + 1) // 2) % MOD
print("Ví dụ, edge case, 1500 test ngẫu nhiên, test lớn (có modulo): OK")

Ví dụ, edge case, 1500 test ngẫu nhiên, test lớn (có modulo): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm chuỗi con **khác nhau về nội dung** (dùng `set`)

In [20]:
def wrong_distinct(s):
    seen = set()
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if len(set(t)) == 1:
                seen.add(t)
    return len(seen)

s, exp = examples[0]
print("sai :", wrong_distinct(s), "| đúng:", exp)

sai : 7 | đúng: 13


**Vì sao sai:** đề đếm theo **vị trí** — `"a"` xuất hiện 3 lần được đếm 3. **Sửa:** không khử trùng.

### Lỗi 2 — Gộp các khối cùng ký tự không liền nhau

In [21]:
from collections import Counter
def wrong_count_chars(s):
    return sum(c * (c + 1) // 2 for c in Counter(s).values()) % MOD

s, exp = examples[0]
print("sai :", wrong_count_chars(s), "| đúng:", exp)

sai : 15 | đúng: 13


**Vì sao sai:** hai khối `a` ở đầu và cuối không liền nhau; `Counter` coi như một khối dài 3 ⇒ đếm thừa `"aa"`, `"aaa"` không tồn tại. **Sửa:** đếm theo **khối liên tiếp**.

### Lỗi 3 — Quên cộng khối cuối cùng (vòng lặp chỉ cộng khi gặp ký tự mới)

In [22]:
def wrong_last_block(s):
    ans, L = 0, 1
    for j in range(1, len(s)):
        if s[j] == s[j - 1]:
            L += 1
        else:
            ans += L * (L + 1) // 2; L = 1
    return ans % MOD                              # SAI: thiếu khối cuối

print([(s, wrong_last_block(s), exp) for s, exp in examples])

[('abbcccaa', 10, 13), ('xy', 1, 2), ('zzzzz', 0, 15)]


**Vì sao sai:** khối cuối không có "ký tự mới" theo sau để kích hoạt việc cộng. **Sửa:** sau vòng lặp cộng thêm `L(L+1)/2`, hoặc dùng Cách 3 (cộng `cur` tại mỗi bước).

### Lỗi 4 — Dùng `/` (chia số thực) trong công thức

In [23]:
def wrong_float(s):
    return int(sum(L * (L + 1) / 2 for L in (len(list(g)) for _, g in groupby(s)))) % MOD

s = "a" * 10 ** 5 + "b" * 99999 + "c" * 77777
print("số thực:", wrong_float(s), "| số nguyên:", Solution().countHomogenous(s))
x = 134217729 * 134217730
print("ví dụ mất chính xác:", x // 2, "vs", int(x / 2))

số thực: 24669662 | số nguyên: 24669662
ví dụ mất chính xác: 9007199456067585 vs 9007199456067584


**Vì sao sai (tiềm ẩn):** `/` cho `float` có 53 bit định trị; với số lớn (trên `2⁵³ ≈ 9·10¹⁵`) sẽ mất chính xác. Ở giới hạn `10⁵` thì `L(L+1)/2 ≈ 5·10⁹` vẫn đúng, nhưng thói quen này gây lỗi ở bài khác. **Sửa:** luôn dùng `//` với số nguyên.

### Lỗi 5 — Quên modulo

In [24]:
def wrong_no_mod(s):
    ans = cur = 0; prev = ""
    for ch in s:
        cur = cur + 1 if ch == prev else 1; prev = ch; ans += cur
    return ans

print("không mod:", wrong_no_mod("a" * 10 ** 5), "| có mod:", Solution().countHomogenous("a" * 10 ** 5))

không mod: 5000050000 | có mod: 49965


**Vì sao sai:** `5000050000 > 10⁹ + 7`, đề yêu cầu trả số đã lấy mod. **Sửa:** `% MOD`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm chuỗi con thỏa tính chất **đóng theo chuỗi con**" (chuỗi con của chuỗi đạt cũng đạt) ⇒ đếm theo **điểm kết thúc** với biến độ dài hợp lệ dài nhất `cur`.
- Ký tự giống nhau liên tiếp ⇒ **run-length** / `groupby`.
- Khối dài `L` ⇒ `L(L+1)/2` chuỗi con.
- Nhớ modulo và dùng `//`.

## 15. Biến thể của bài

1. **Chỉ đếm chuỗi con đồng nhất độ dài ≥ k**: mỗi khối đóng góp `Σ_{t=k}^{L} (L − t + 1) = (L−k+1)(L−k+2)/2` nếu `L ≥ k`.
2. **413 Arithmetic Slices**: cùng mẫu "đếm theo điểm kết thúc" với điều kiện hiệu bằng nhau.
3. **1513 Number of Substrings With Only 1s** — gần như giống hệt (chỉ ký tự `'1'`).
4. **2348 Number of Zero-Filled Subarrays** — mảng số, đếm đoạn toàn 0.
5. **Cho phép đổi tối đa một ký tự** để kéo dài khối — Bài tập 5 dạng khác (khó hơn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | Kiểm từng chuỗi con | `O(n³)` | `O(n)` | Kiểm chứng |
| 1'. Mở rộng | Từ mỗi `i` đi tới hết khối | `O(n²)` | `O(1)` | Trung gian |
| 2. Khối + công thức | `Σ L(L+1)/2` | `O(n)` | `O(1)` | Dễ hiểu |
| 3. Biến `cur` | Đếm theo điểm kết thúc | `O(n)` | `O(1)` | **Bản nộp**, tổng quát hóa tốt |
| 4. `groupby` | Như 2, viết gọn | `O(n)` | `O(1)` | Code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"aaa"`; (b) `"abcd"`; (c) `"aabbaa"`; (d) `"abbbbbbc"`.

**Bài 2 (tính tay):** Với `s = "ppqqqppp"`, liệt kê các khối, tính đáp án, và trace `cur` tại từng vị trí.

**Bài 3 (điền TODO):** Hoàn thiện `run_lengths(s)` trả list độ dài các khối liên tiếp (không dùng `groupby`).

**Bài 4 (tự viết lại):** Viết `count_at_least_k(s, k)` đếm chuỗi con đồng nhất có độ dài `≥ k`.

**Bài 5 (mini challenge):** Viết `count_by_char(s)` trả dict `ký tự → số chuỗi con đồng nhất gồm ký tự đó`; kiểm tra tổng các giá trị bằng đáp án bài gốc (chưa mod).

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def run_lengths(s):
    res = []
    # TODO: hai con trỏ i, j như Cách 2; append(j - i)
    return res

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_at_least_k(s, k):
    # TODO: với mỗi khối L >= k: cộng (L-k+1)*(L-k+2)//2
    pass

def count_by_char(s):
    # TODO: dict mặc định 0, cộng L*(L+1)//2 vào ký tự của khối
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tách khối. Mức 2: áp công thức. Mức 3: (a) `6`; (b) `4`; (c) `3+3+3 = 9`; (d) `1 + 21 + 1 = 23`.

**Bài 2.** Mức 1: khối `pp`, `qqq`, `ppp`. Mức 2: `3 + 6 + 6 = 15`. Mức 3: `cur` = `1,2,1,2,3,1,2,3`, tổng 15.

**Bài 3.** Mức 1: `i = 0`. Mức 2: `while j < n and s[j] == s[i]`. Mức 3: `res.append(j - i); i = j`.

**Bài 4.** Mức 1: khối `L` có `L − t + 1` chuỗi con độ dài `t`. Mức 2: tổng `t = k..L` là `1 + 2 + … + (L−k+1)`. Mức 3: `m = L − k + 1`, cộng `m(m+1)/2` nếu `m > 0`. Kiểm bằng vét cạn.

**Bài 5.** Mức 1: `groupby` hoặc `run_lengths` kèm ký tự. Mức 2: `d[ch] += L*(L+1)//2`. Mức 3: `sum(d.values()) == count`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
b1 = [Solution().countHomogenous(s) for s in ["aaa", "abcd", "aabbaa", "abbbbbbc"]]
print("Bài 1:", b1); assert b1 == [6, 4, 9, 23]
print("Bài 2:"); assert trace_streak("ppqqqppp") == 15

Bài 1: [6, 4, 9, 23]
Bài 2:
j=0 s[j]='p' giống trước=False cur=1 ans=  1 | chuỗi con kết thúc tại j: ['p']
j=1 s[j]='p' giống trước=True  cur=2 ans=  3 | chuỗi con kết thúc tại j: ['p', 'pp']
j=2 s[j]='q' giống trước=False cur=1 ans=  4 | chuỗi con kết thúc tại j: ['q']
j=3 s[j]='q' giống trước=True  cur=2 ans=  6 | chuỗi con kết thúc tại j: ['q', 'qq']
j=4 s[j]='q' giống trước=True  cur=3 ans=  9 | chuỗi con kết thúc tại j: ['q', 'qq', 'qqq']
j=5 s[j]='p' giống trước=False cur=1 ans= 10 | chuỗi con kết thúc tại j: ['p']
j=6 s[j]='p' giống trước=True  cur=2 ans= 12 | chuỗi con kết thúc tại j: ['p', 'pp']
j=7 s[j]='p' giống trước=True  cur=3 ans= 15 | chuỗi con kết thúc tại j: ['p', 'pp', 'ppp']


In [28]:
# Bài 3, 4, 5
def run_lengths(s):
    res, i, n = [], 0, len(s)
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        res.append(j - i); i = j
    return res

def count_at_least_k(s, k):
    total = 0
    for L in run_lengths(s):
        m = L - k + 1
        if m > 0:
            total += m * (m + 1) // 2
    return total

def count_by_char(s):
    d = {}
    for ch, g in groupby(s):
        L = sum(1 for _ in g)
        d[ch] = d.get(ch, 0) + L * (L + 1) // 2
    return d

rng = random.Random(8)
for _ in range(800):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 20)))
    assert run_lengths(s) == [len(list(g)) for _, g in groupby(s)]
    k = rng.randint(1, 4)
    brute = sum(1 for i in range(len(s)) for j in range(i, len(s)) if j - i + 1 >= k and len(set(s[i:j + 1])) == 1)
    assert count_at_least_k(s, k) == brute
    assert sum(count_by_char(s).values()) == count_streak(s)
print("Bài 3, 4, 5: OK |", run_lengths("abbcccaa"), count_at_least_k("abbcccaa", 2), count_by_char("abbcccaa"))

Bài 3, 4, 5: OK | [1, 2, 3, 2] 5 {'a': 4, 'b': 3, 'c': 6}


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm theo điểm kết thúc":**

```python
ans = cur = 0; prev = None
for ch in s:
    cur = cur + 1 if ch == prev else 1
    prev = ch
    ans = (ans + cur) % MOD
return ans
```

**Checklist:**
- [ ] Đếm theo **vị trí** (không khử trùng nội dung)?
- [ ] Khối là các ký tự **liên tiếp** (không gộp các khối cùng chữ cách xa nhau)?
- [ ] Không quên khối cuối?
- [ ] Dùng `//` và lấy `% (10**9 + 7)`?
- [ ] Đã test `n = 1`, toàn một ký tự `10⁵`, xen kẽ `abab…`?